In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%run /Workspace/fmcg_domain/databricks_fmcg_data_engineering/Setup/utilities

In [0]:
bronze_schema,silver_schema,gold_schema
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("datasource","customers","Data source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("datasource")

In [0]:
import os 
base_s3_path = f"s3://databrickssaugat/{data_source}/"
s3_path = f"s3://databrickssaugat/{data_source}/landing/*.csv"

s3_path

In [0]:
df = spark.read\
    .format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load(s3_path)\
    .withColumn("ingest_timestamp", current_timestamp())\
    .select("*" , "_metadata.file_name", "_metadata.file_size")


display(df)

In [0]:
bronze_customers_table = f"{catalog}.{bronze_schema}.{data_source}"
bronze_customers_table

In [0]:

df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(bronze_customers_table)

In [0]:
df.select("file_name").distinct().collect()

In [0]:
files = [x.file_name for x in df.select("file_name").distinct().collect()]

for file in files:
    
    dbutils.fs.mv(f"{base_s3_path}/landing/{file}", f"{base_s3_path}/processed/{file}")